# Chapter 6: Finetuning for Text Classification

This notebook covers finetuning pretrained models for downstream tasks:
- Classification head design
- Transfer learning from pretraining
- Finetuning strategies
- Evaluation metrics

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 6.1: Classification Dataset

In [ ]:
# Sample classification data (sentiment analysis)
sample_data = [
    ("This movie is absolutely amazing! I loved every second of it.", 1),
    ("Terrible film. Waste of time and money. Don't watch.", 0),
    ("Great acting and storyline. Highly recommended!", 1),
    ("Boring and predictable. Very disappointed.", 0),
    ("Outstanding performance by the actors.", 1),
    ("Not worth watching. Poor quality production.", 0),
    ("Fantastic! One of the best movies I've ever seen.", 1),
    ("Awful. Could not finish watching it.", 0),
    ("Excellent direction and cinematography.", 1),
    ("Disappointing and underwhelming experience.", 0),
]

# In practice, use larger datasets like IMDB, SST-2, etc.

class ClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Tokenize
        tokens = self.tokenizer.encode(text)
        
        # Truncate or pad
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]
        else:
            tokens = tokens + [50256] * (self.max_length - len(tokens))  # Padding token
        
        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

print("ClassificationDataset implemented")

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

# Split data
texts = [item[0] for item in sample_data]
labels = [item[1] for item in sample_data]

split_idx = int(0.8 * len(texts))
train_texts, val_texts = texts[:split_idx], texts[split_idx:]
train_labels, val_labels = labels[:split_idx], labels[split_idx:]

# Create datasets
train_dataset = ClassificationDataset(train_texts, train_labels, tokenizer)
val_dataset = ClassificationDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples: {len(val_dataset)}")

# Test a batch
batch = next(iter(train_loader))
print(f"\nBatch input_ids shape: {batch['input_ids'].shape}")
print(f"Batch labels: {batch['label']}")

## 6.2: Classification Head

## 6.3: Model for Finetuning

## 6.4: Finetuning Loop

## 6.5: Finetuning Strategies

# Example: Layer-wise unfreezing
def progressive_unfreezing(model, layer_idx):
    """
    Gradually unfreeze layers during finetuning.
    Start with only classification head, then unfreeze layers.
    """
    for i, module in enumerate(model.gpt_model.blocks):
        if i >= layer_idx:
            for param in module.parameters():
                param.requires_grad = True
        else:
            for param in module.parameters():
                param.requires_grad = False

print("Progressive unfreezing function defined")

# Confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(example_labels, example_predictions)
print(f"\nConfusion Matrix:")
print(f"  TN: {cm[0,0]} | FP: {cm[0,1]}")
print(f"  FN: {cm[1,0]} | TP: {cm[1,1]}")

## Summary

Finetuning for classification involves:
1. Add classification head to pretrained model
2. Choose finetuning strategy (full, layer-wise, LoRA, etc.)
3. Use appropriate learning rates and regularization
4. Evaluate with task-specific metrics

This approach, called transfer learning, achieves strong results with limited task-specific data!